<div style="text-align:center;">
  <img src="https://github.com/LinkedEarth/Logos/blob/master/PaleoPAL/PaleoPal_rectangular_light.png?raw=true" width="500">
</div>

## PaleoPAL Evaluation: Notebook 1

This notebook is part of a series of evaluation tests for the [PaleoPAL](linked.earth/paleopal) assistant. You have two hours to complete the assignment. 

The notebook is divided into the following sections:
1. Data Gathering (1 hour and 15min)
2. Analysis (30 min)
3. Visualization (5min)

If you cannot complete the assignment for each section in the time alloted, use the solution and move on to the next section. 

**Use VS Code to complete the assignment**. 

In [28]:
#### Import libraries

## Data Gathering (1 hour and 15min)

In this part of the assignment, you will gather three marine sediment Mg/Ca records from three different data sources: a SPARQL endpoint to the LiPDGraph, PANGAEA through PyleoTUPS, and a local LiPD file using PyLiPD.

After each block, confirm that the resulting dataframe contains age or time values, Mg/Ca values, location metadata, archive and/or proxy metadata and enough other descriptive metadata to build a Pyleoclim Geoseries later.

### LiPDGraph

Using a SPARQL Query on the LiPDGraph, look for the Dataset named `MD98_2161.Fan.2018`.

**Assignment task:** 
a. Query the LiPDGraph endpoint (https://linkedearth.graphdb.mint.isi.edu/repositories/LiPDVerse-dynamic) for the dataset `MD98_2161.Fan.2018`. Your query should return latitude, longitude, archive type, paleo variable name (set as `Mg/Ca`), TSID, Mg/Ca values, Mg/Ca units, proxy, time or age variable name, time TSID, time or age values, and time units. **Make sure that you restrict the query to `MD98_2161.Fan.2018`. 
b. Convert the query response into a `pandas.DataFrame`.
c. Use the original age variable when multiple age or time columns are available. (*Hint: Look at the time TSID*)
d. Remove duplicate rows from the resulting DataFrame so that **one** clean record remains for later analysis.

In [30]:
query = """

PREFIX le: <http://linked.earth/ontology#>
PREFIX le_var: <http://linked.earth/ontology/paleo_variables#>
PREFIX wgs84: <http://www.w3.org/2003/01/geo/wgs84_pos#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?dataSetName ?archiveType ?geo_meanLat ?geo_meanLon
    ?paleoData_variableName ?paleoData_TSID ?paleoData_values ?paleoData_units ?paleoData_proxy
    ?time_variableName ?time_TSID ?time_values ?time_units
WHERE {
    ?ds a le:Dataset .
    ?ds le:hasName ?dataSetName .
    VALUES ?dataSetName {"MD98_2161.Fan.2018"} .

    OPTIONAL { ?ds le:hasArchiveType ?atObj . ?atObj rdfs:label ?archiveType . }

    ?ds le:hasLocation ?loc .
    OPTIONAL { ?loc wgs84:lat  ?geo_meanLat . }
    OPTIONAL { ?loc wgs84:long ?geo_meanLon . }

    ?ds le:hasPaleoData ?data .
    ?data le:hasMeasurementTable ?table .

    ?table le:hasVariable ?var .
    ?var le:hasName ?paleoData_variableName .
    FILTER(REGEX(?paleoData_variableName, "mg.?ca", "i"))
    ?var le:hasValues ?paleoData_values .
    OPTIONAL { ?var le:hasVariableId ?paleoData_TSID . }
    OPTIONAL { ?var le:hasUnits ?uObj . ?uObj rdfs:label ?paleoData_units . }
    OPTIONAL { ?var le:hasProxy ?pObj . ?pObj rdfs:label ?paleoData_proxy . }

    ?table le:hasVariable ?timevar .
    ?timevar le:hasName ?time_variableName .
    ?timevar le:hasValues ?time_values .
    OPTIONAL { ?timevar le:hasVariableId ?time_TSID . }
    OPTIONAL { ?timevar le:hasUnits ?tuObj . ?tuObj rdfs:label ?time_units . }
    ?timevar le:hasStandardVariable le_var:age .
}
"""

url = 'https://linkedearth.graphdb.mint.isi.edu/repositories/LiPDVerse-dynamic'

response = requests.post(url, data = {'query': query})

data = io.StringIO(response.text)
df = pd.read_csv(data, sep=",")

df = df[df['time_TSID'] == 'T2L_MD98_2161_age_original']

df1 = df.drop_duplicates(subset=['paleoData_values', 'time_values']).reset_index(drop=True)

### PyleoTUPS

Using PyleoTUPS, Open the [PANGAEA Dataset `MD01_2378.Xu.2008`](https://doi.pangaea.de/10.1594/PANGAEA.831197) and retrieve its study summary, site metadata, and data table.

**Assignment task:** 

a. Search for the study using the information from the [PANGAEA page](https://doi.pangaea.de/10.1594/PANGAEA.831197).
b. Display the study summary so you can verify the citation and dataset identity.
c. Display the geographic metadata so you can recover the site name, latitude, and longitude.
d. Load the data table and inspect the first rows. Make note of the column names and how they can be used later.

In [31]:
ds = pt.PangaeaDataset()
res = ds.search_studies(study_ids = '831197')

df_summary = ds.get_summary()
display(df_summary)

df_geo = ds.get_geo()
display(df_geo)

df_data = ds.get_data(study_id = '831197')[0]
display(df_data.head())

### PyLiPD

Load a local LiPD file (`MD98_2176.Stott.2007.lpd`) and extract the Mg/Ca time series from it. 

**Assignment task:**
a. Open the dataset and retrieve relevant timeseries information
b. Filter the dataframe to keep only rows where the paleo variable is Mg/Ca.


In [1]:
D = LiPD()

data_path = 'MD98_2176.Stott.2007.lpd'
D.load(data_path)

names = D.get_all_dataset_names()
print(names)

ts_list, df = D.get_timeseries(D.get_all_dataset_names(), to_dataframe=True)
df

df3 = df[df['paleoData_variableName'] == 'Mg/Ca']
df3

## Analysis (30 mins)

In this part of the assignment, you will convert the three gathered records into Pyleoclim `GeoSeries` objects and compare two of them with wavelet coherence.


### Create Pyleoclim GeoSeries Objects

Convert each dataframe-based record into a `pyleoclim.GeoSeries` so the records share a common analysis structure. Use the DataFrame obtained at the end of each data loading (i.e., you should only have one timeseries by the end of the filtering).

**Assignment task:**
a. For each of the DataFrame produced in the previous section, create a `pyleoclim.GeoSeries` object that contains the following information: time name, time units, value name, value units, label, archive type, latitude, and longitude
b. Make a simple plot for each `GeoSeries` to inspect the record visually.

<div style="background-color:#e6f2ff; border-left:5px solid #2f80ed; padding:12px 16px; margin-top:16px; border-radius:6px;">
  <strong>Note:</strong> A few things to keep in mind as you create your GeoSeries: a. LiPDGraph encodes data as a string (not an array), b. watch your units!
</div>


In [33]:
t = np.asarray(ast.literal_eval(df1['time_values'].iloc[0]), dtype=float)
v = np.asarray(ast.literal_eval(df1['paleoData_values'].iloc[0]), dtype=float)

print(t.shape, v.shape)   

ts_1 = pyleo.GeoSeries(
    time=t, value=v,
    time_name='Age', time_unit='yr BP',
    value_name='Mg/Ca', value_unit='mmol/mol',
    label='MD98_2161 Fan 2018',
    archiveType='Marine sediment',
    lat=df1['geo_meanLat'].iloc[0],   # -5.21
    lon=df1['geo_meanLon'].iloc[0],   # 117.48
)

ts_1.plot()

ts_2 = pyleo.GeoSeries(time=df_data['Age']*1000, value=df_data['G. ruber w Mg/Ca'],
                     time_name='Age', time_unit='yr BP',
                     value_name='Mg/Ca', value_unit='mmol/mol',
                     label='MD01-2378 Xu 2008',
                     archiveType='Marine Sediment',
                     lat=df_data['Latitude'].iloc[0],
                     lon=df_data['Longitude'].iloc[0])


ts_2.plot()

ts_3 = pyleo.GeoSeries(time=df3['age'].iloc[0], value=df3['paleoData_values'].iloc[0],
                     time_name='Age', time_unit='yr BP',
                     value_name=df3['paleoData_variableName'].iloc[0],
                     label=df3['dataSetName'].iloc[0],
                     archiveType=df3['archiveType'].iloc[0],
                     lat=df3['geo_meanLat'].iloc[0],
                     lon=df3['geo_meanLon'].iloc[0])

ts_3.plot()

### Wavelet Coherence

Compare two Mg/Ca records using wavelet coherence and visualize the result.

**Assignment task:**
a. On our three Geoseries, run three coherence tests (1 v 2, 1 v 3, 2 v 3) **without** significance testing. 

In [34]:
coh_1v2 = ts_1.wavelet_coherence(ts_2)
coh_1v3 = ts_1.wavelet_coherence(ts_3)
coh_2v3 = ts_2.wavelet_coherence(ts_3)

## Visualization (5 mins)

**Assignment task:**
Plot each of the results of the Wavelet Coherence Analysis. 

In [35]:
coh_1v2.plot()
coh_1v3.plot()
coh_2v3.plot()